In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import joblib

In [21]:
# Load the original data
data_original = pd.read_excel('Kaggle_Sirio_Libanes_ICU_Prediction.xlsx')
data_original.head(3)

,PATIENT_ID,AGE_ABOVE65,AGE_PERCENTIL,GENDER,DISEASE GROUPING 1,DISEASE GROUPING 2,DISEASE GROUPING 3,DISEASE GROUPING 4,DISEASE GROUPING 5,DISEASE GROUPING 6,...,TEMPERATURE_DIFF,OXYGEN_SATURATION_DIFF,BLOODPRESSURE_DIASTOLIC_DIFF_REL,BLOODPRESSURE_SISTOLIC_DIFF_REL,HEART_RATE_DIFF_REL,RESPIRATORY_RATE_DIFF_REL,TEMPERATURE_DIFF_REL,OXYGEN_SATURATION_DIFF_REL,WINDOW,ICU
0,0,1,60th,0,0.0,0.0,0.0,0.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0-2,0
1,0,1,60th,0,0.0,0.0,0.0,0.0,1.0,1.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,2-4,0
2,0,1,60th,0,0.0,0.0,0.0,0.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4-6,0


In [22]:
# Separate features and target
X_original = data_original.drop(columns=['ICU'])
y_original = data_original['ICU']

# Split into training/validation (285 patients) and final testing (100 patients)
X_train_val, X_test_final, y_train_val, y_test_final = train_test_split(
    X_original, y_original, test_size=100, stratify=y_original, random_state=42)

# Display the shapes of the resulting datasets to ensure correct splitting
X_train_val.shape, X_test_final.shape, y_train_val.shape, y_test_final.shape

((1825, 230), (100, 230), (1825,), (100,))

In [23]:
#Test original data using Randomforest

# Define the preprocessing and modeling pipeline for original data
preprocessor_original = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', X_train_val.select_dtypes(include=['float64', 'int64']).columns),
        ('cat', OneHotEncoder(drop='first'), X_train_val.select_dtypes(include=['object']).columns)
    ])

model_pipeline_original = Pipeline(steps=[
    ('preprocessor', preprocessor_original),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the model on the original data
model_pipeline_original.fit(X_train_val, y_train_val)

# Evaluate on final test set
y_test_final_pred_original = model_pipeline_original.predict(X_test_final)
accuracy_original = accuracy_score(y_test_final, y_test_final_pred_original)
accuracy_original

0.92

In [24]:
#Test imputed (filled-in) data with Random forest

# List of binomial columns
binomial_cols = ['AGE_ABOVE65', 'GENDER', 'DISEASE GROUPING 1', 'DISEASE GROUPING 2', 
                 'DISEASE GROUPING 3', 'DISEASE GROUPING 4', 'DISEASE GROUPING 5', 
                 'DISEASE GROUPING 6', 'HTN', 'IMMUNOCOMPROMISED', 'OTHER']

# Define the preprocessing and modeling pipeline with correct imputation strategies
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='mean'))]), 
         [col for col in X_train_val.select_dtypes(include=['float64', 'int64']).columns if col not in binomial_cols]),
        ('bin', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))]), binomial_cols),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), 
                                ('encoder', OneHotEncoder(drop='first'))]), 
         [col for col in X_train_val.select_dtypes(include=['object']).columns if col not in binomial_cols])
    ])

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the model
model_pipeline.fit(X_train_val, y_train_val)

# Evaluate on final test set
y_test_final_pred = model_pipeline.predict(X_test_final)
accuracy = accuracy_score(y_test_final, y_test_final_pred)

accuracy

0.92

In [25]:
#Test a Logistic Regression model on the imputed data

# Define the preprocessing and modeling pipeline for Logistic Regression
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Train the Logistic Regression model
logistic_pipeline.fit(X_train_val, y_train_val)

# Evaluate on final test set
y_test_final_pred_logistic = logistic_pipeline.predict(X_test_final)
accuracy_logistic = accuracy_score(y_test_final, y_test_final_pred_logistic)

accuracy_logistic

0.93

In [26]:
#Test a decision tree model on the imputed data

from sklearn.tree import DecisionTreeClassifier

# Define the preprocessing and modeling pipeline for Decision Tree
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Train the Decision Tree model
tree_pipeline.fit(X_train_val, y_train_val)

# Evaluate on final test set
y_test_final_pred_tree = tree_pipeline.predict(X_test_final)
accuracy_tree = accuracy_score(y_test_final, y_test_final_pred_tree)

accuracy_tree

0.86

In [27]:
#Tune logistic regression

# Define a simplified parameter grid for Logistic Regression
param_grid_simplified = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear'],
    'classifier__max_iter': [200, 500]  # Reduced number of iterations
}

# Define the GridSearchCV with the simplified parameter grid
grid_search_simplified = GridSearchCV(logistic_pipeline, param_grid_simplified, cv=5, scoring='accuracy')

# Perform GridSearchCV to find the best parameters
grid_search_simplified.fit(X_train_val, y_train_val)

# Get the best parameters and best score
best_params_simplified = grid_search_simplified.best_params_
best_score_simplified = grid_search_simplified.best_score_

best_params_simplified, best_score_simplified



({'classifier__C': 1,
  'classifier__max_iter': 200,
  'classifier__penalty': 'l2',
  'classifier__solver': 'liblinear'},
 0.8504109589041097)

In [28]:
# Define the Logistic Regression pipeline with the best parameters
logistic_pipeline_optimized = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=1, penalty='l2', solver='liblinear', random_state=42, max_iter=200))
])

# Train the optimized Logistic Regression model
logistic_pipeline_optimized.fit(X_train_val, y_train_val)

# Evaluate on final test set
y_test_final_pred_optimized = logistic_pipeline_optimized.predict(X_test_final)
accuracy_optimized = accuracy_score(y_test_final, y_test_final_pred_optimized)

accuracy_optimized

0.93

In [29]:
# Save the trained model to a file
model_filename = 'logistic_regression_capstone.pkl'
joblib.dump(logistic_pipeline_optimized, model_filename)

['logistic_regression_capstone.pkl']

In [31]:
#Test on full data set
loaded_model = joblib.load('logistic_regression_capstone.pkl')

df = pd.read_excel('Kaggle_Sirio_Libanes_ICU_Prediction.xlsx')

X_new = df.drop(columns=['ICU'])
y_new = df['ICU']

y_new_pred = loaded_model.predict(X_new)
accuracy_new = accuracy_score(y_new, y_new_pred)

accuracy_new

0.8768831168831169